# Ceeria Workflow 개선 사항 정리
## Multi-turn Memory · HyDE · RAGAS 평가 파이프라인

오늘 `workflow_notebook_toolcalling.py`에 추가된 세 가지 기능과  
별도 파일 `ragas_eval.py`로 만든 평가 파이프라인을 설명합니다.

---

### 개선 전 / 후 그래프 흐름

```
[개선 전]
START → classify → [fetch_* / retrieve] → generate_response → END

[개선 후]
START → resolve_query → classify → [fetch_* / retrieve(HyDE)] → generate_response → END
                                                                         ↑
                                                               chat_history 주입 + 업데이트
```

| 추가 기능 | 핵심 변경 파일 | 효과 |
|---|---|---|
| Multi-turn Memory | `workflow_notebook_toolcalling.py` | 대화 이력 유지, 지칭어 해소 |
| HyDE | `workflow_notebook_toolcalling.py` | RAG 검색 품질 향상 |
| RAGAS 평가 | `ragas_eval.py` (신규) | RAG 품질 자동 측정 |

---
## 1. Multi-turn Memory (다중턴 대화 기억)

### 문제
기존 워크플로우는 매 질문이 독립적으로 처리되었습니다.  
엔지니어가 "그 장비 모델이 뭐야?" 라고 물어도, 직전에 조회한 장비를 기억하지 못합니다.

### 해결 방법
세 가지를 추가했습니다.

| 추가 요소 | 역할 |
|---|---|
| `WorkflowState.chat_history` | 대화 이력 저장 필드 |
| `resolve_query` 노드 | 지칭어("그거", "해당 장비") 를 LLM으로 구체적인 값으로 교체 |
| `MemorySaver` checkpointer | `thread_id` 별로 state 전체를 인메모리 유지 |

### 동작 흐름
```
턴 1: "M15A001 지금 돌아가?"
  → resolve_query: 이력 없음 → 원본 그대로
  → classify: EQP 판별
  → generate_response: 답변 후 chat_history에 Q&A 추가

턴 2: "그 장비 모델이 뭐야?"
  → resolve_query: 이력 있음 + "그 장비" 감지
               → LLM 재작성 → "M15A001 모델이 뭐야?"
  → classify: EQP 판별 (M15A001)
  → generate_response: 정확한 답변
```

In [ ]:
## ── [변경 1-A] WorkflowState에 필드 2개 추가 ─────────────────────────────
from typing import TypedDict, Optional, List

class WorkflowState(TypedDict, total=False):
    query: str
    resolved_query: str   # ★ NEW: 지칭어가 해소된 질문 (resolve_query 노드가 채움)
    answer: str
    # ... (기존 필드들 생략)
    chat_history: List[dict]  # ★ NEW: [{"role":"user","content":"..."}, {"role":"assistant",...}]

In [ ]:
## ── [변경 1-B] resolve_query 노드 (신규 추가) ────────────────────────────
#
# 핵심 설계:
#   - 지칭어가 없으면 LLM 호출 없이 즉시 통과 (비용 0)
#   - 있을 때만 최근 3턴(6개 메시지)을 context로 LLM에 전달

_AMBIGUOUS_TOKENS = ["그거", "그 장비", "그 랏", "해당", "거기", "그게", "그것", "그 모델", "동일한"]

def resolve_query(state: WorkflowState) -> dict:
    history = state.get("chat_history") or []
    query   = state["query"]

    # 지칭어 없거나 이력 없으면 → 아무것도 안 함 (classify가 원본 query 사용)
    if not history or not any(t in query for t in _AMBIGUOUS_TOKENS):
        return {}

    recent      = history[-6:]   # 최근 3턴만 사용
    history_txt = "\n".join(f"{m['role']}: {m['content']}" for m in recent)

    response = llm.invoke([
        SystemMessage(content=(
            "이전 대화를 참고하여 아래 질문의 지칭어(그거, 해당, 그 장비 등)를 "
            "구체적인 값으로 바꿔서 한 문장으로 재작성하세요. 질문만 출력하세요.\n\n"
            f"[이전 대화]\n{history_txt}"
        )),
        HumanMessage(content=query),
    ])
    resolved = response.content.strip()
    print(f"[resolve_query] '{query}' → '{resolved}'")
    return {"resolved_query": resolved}   # state에 저장, classify가 우선 사용

In [ ]:
## ── [변경 1-C] generate_response — 대화 이력 주입 + 업데이트 ─────────────
from langchain_core.messages import AIMessage   # 추가 import

def generate_response(state: WorkflowState) -> dict:
    # ... context 구성 (기존 코드 생략) ...

    history       = state.get("chat_history") or []
    current_query = state.get("resolved_query") or state["query"]

    # 시스템 메시지 + 최근 대화 이력(최대 6개) + 현재 질문
    messages = [SystemMessage(content=_RAG_PROMPT.format(context=context))]
    for msg in history[-6:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))   # ★ AIMessage 추가
    messages.append(HumanMessage(content=current_query))

    response = llm.invoke(messages)

    # ★ 이번 턴 Q&A를 이력에 추가해서 반환
    updated_history = history + [
        {"role": "user",      "content": current_query},
        {"role": "assistant", "content": response.content},
    ]
    return {"answer": response.content, "chat_history": updated_history}

In [ ]:
## ── [변경 1-D] build_graph — resolve_query 노드 + MemorySaver ────────────
from langgraph.checkpoint.memory import MemorySaver   # ★ 추가 import

def build_graph():
    wb = StateGraph(WorkflowState)

    wb.add_node("resolve_query", resolve_query)   # ★ 신규 노드
    wb.add_node("classify",      classify)
    # ... 나머지 노드 동일 ...

    wb.add_edge(START,           "resolve_query")  # ★ START → resolve_query 로 변경
    wb.add_edge("resolve_query", "classify")       # ★ resolve_query → classify

    # ... 나머지 엣지 동일 ...

    return wb.compile(checkpointer=MemorySaver())  # ★ checkpointer 추가


## ── [변경 1-E] run_workflow — thread_id 파라미터 추가 ───────────────────
def run_workflow(query: str, thread_id: str = "default") -> dict:
    """
    thread_id가 같으면 이전 대화 이력(chat_history)이 유지됩니다.
    새 대화를 시작하려면 다른 thread_id를 사용하세요.
    서버 재시작 시 MemorySaver는 초기화됩니다.
    (영속성이 필요하면 SqliteSaver / PostgresSaver 로 교체)
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = workflow_app.invoke({"query": query}, config=config)
    return result


# ── 사용 예시 ──────────────────────────────────────────────────────────────
# run_workflow("M15A001 지금 돌아가?",     thread_id="session-A")
# run_workflow("그 장비 모델이 뭐야?",      thread_id="session-A")  # ← 이력 유지
# run_workflow("M15에 그 모델 몇 대 있어?", thread_id="session-A")  # ← 이력 유지

---
## 2. HyDE (Hypothetical Document Embeddings)

### 문제
엔지니어의 구어체 질문과 전문 기술 문서는 **임베딩 벡터 공간에서 거리가 멀어** 관련 문서를 못 찾는 경우가 생깁니다.

```
사용자: "LPCVD 두께 왜 흔들려?"   → 임베딩 벡터 A
실제문서: "LPCVD 두께 편차의 주요 원인은 가스 유량 불균형..."  → 임베딩 벡터 B
A와 B 사이 거리가 멀어 검색 실패
```

### 해결 방법
질문으로 직접 검색하지 않고, **LLM이 먼저 가상 답변을 생성한 뒤 그 벡터로 검색**합니다.

```
질문: "LPCVD 두께 왜 흔들려?"
    ↓ LLM (_HYDE_PROMPT)
가상 문서: "LPCVD 두께 편차의 주요 원인은 가스 유량 불균형, 챔버 온도 불균일..."
    ↓ 임베딩
→ 실제 문서와 벡터 거리 단축 → 검색 recall 향상
```

### 중요한 설계 결정

| 단계 | 사용 텍스트 | 이유 |
|---|---|---|
| 벡터 검색 | 가상 문서 임베딩 | 전문 용어로 실제 문서와 거리 단축 |
| 키워드 재랭킹 | 원본 질문 기준 | 사용자 의도 키워드("두께", "흔들려") 유지 |

### 적용 범위
`skip_rag=False`인 **general 경로에서만** 작동합니다.  
LOT/EQP/FAB/OPER 구조적 경로는 HyDE LLM 호출이 발생하지 않습니다.

In [ ]:
## ── [변경 2] retrieve 노드에 HyDE 추가 ──────────────────────────────────

# ① 가상 문서 생성용 프롬프트
_HYDE_PROMPT = (
    "당신은 반도체 FAB 공정 전문가입니다. "
    "아래 질문에 대한 답을 실제 기술 문서처럼 전문 용어를 사용해 150자 이내로 작성하세요. "
    "문서 형식으로만 출력하고 추가 설명은 하지 마세요."
)

def _generate_hypothetical_doc(query: str) -> str:
    """질문 → 가상 기술 문서 생성 (HyDE 핵심)"""
    response = llm.invoke([
        SystemMessage(content=_HYDE_PROMPT),
        HumanMessage(content=query),
    ])
    return response.content.strip()


SIMILARITY_THRESHOLD = 0.50   # cosine similarity 기준

def retrieve(state: WorkflowState) -> dict:
    if state.get("skip_rag", True):
        print("[retrieve] skip_rag=True → 스킵")
        return {"retrieved_docs": []}

    original_query = state.get("resolved_query") or state["query"]

    # ② HyDE: 가상 문서 생성 → 그 임베딩으로 검색
    hypothetical_doc = _generate_hypothetical_doc(original_query)
    print(f"[retrieve/HyDE] 가상 문서: {hypothetical_doc[:80]}…")
    docs_with_score = vectorstore.similarity_search_with_score(hypothetical_doc, k=5)

    # ③ FAISS L2 distance → cosine similarity 변환 (단위 벡터 기준)
    #    text-embedding-3-small은 정규화된 벡터이므로: cosine = 1 - L2² / 2
    filtered = []
    for doc, score in docs_with_score:
        cosine_sim = max(0.0, 1.0 - (score ** 2) / 2.0)
        if cosine_sim >= SIMILARITY_THRESHOLD:
            doc.metadata["similarity"] = round(cosine_sim, 4)
            filtered.append(doc)

    # ④ 키워드 재랭킹은 원본 질문 기준 (사용자 의도 유지)
    if filtered:
        stop_words = {"은", "는", "의", "가", "을", "를", "에", "와", "과", "로", "이"}
        keywords = [w for w in original_query.split() if len(w) > 1 and w not in stop_words]
        for doc in filtered:
            doc.metadata["keyword_score"] = sum(1 for k in keywords if k.lower() in doc.page_content.lower())
        max_kw = max(doc.metadata["keyword_score"] for doc in filtered) or 1
        for doc in filtered:
            norm = doc.metadata["keyword_score"] / max_kw
            doc.metadata["combined_score"] = doc.metadata["similarity"] * 0.7 + norm * 0.3
        filtered = sorted(filtered, key=lambda x: x.metadata["combined_score"], reverse=True)

    print(f"[retrieve] {len(docs_with_score)}건 검색 → {len(filtered)}건 채택")

    # ⑤ MemorySaver 직렬화 대응: Document 객체 → dict 변환
    #    numpy.float64도 Python float으로 변환
    serializable = [
        {
            "page_content": d.page_content,
            "metadata": {k: float(v) if hasattr(v, "item") else v
                         for k, v in d.metadata.items()},
        }
        for d in filtered
    ]
    return {"retrieved_docs": serializable}

---
## 3. RAGAS 평가 파이프라인 (`ragas_eval.py`)

### RAGAS란?
RAG(Retrieval-Augmented Generation) 품질을 **LLM-as-judge** 방식으로 자동 측정하는 프레임워크입니다.

### 4가지 메트릭

| 메트릭 | 측정 내용 | 필요한 것 |
|---|---|---|
| **Faithfulness** | 답변이 검색 문서에 근거하는가 (hallucination 탐지) | question + answer + contexts |
| **AnswerRelevancy** | 답변이 질문에 맞는가 | question + answer |
| **ContextPrecision** | 검색된 문서 중 실제 관련된 비율 | question + contexts + ground_truth |
| **ContextRecall** | 정답 커버에 필요한 문서를 찾았는가 | contexts + ground_truth |

### 오늘 평가 결과 요약
```
faithfulness       ████████████████░  0.886  ✅
answer_relevancy   ██████████░░░░░░░  0.514  ⚠️  (답변이 질문 의도에 비해 과도하게 긴 경우)
context_precision  ████████████████░  0.837  ✅
context_recall     █████████████████  0.875  ✅
```

### 발견된 주요 이슈
1. **Q6 라우팅 오분류**: "포토 리소그래피에서 패턴 불량이 생기면?" → `oper`로 잘못 분류 → RAG 스킵 → context 0점
2. **Q8 hallucination**: "Future Hold 어떻게 설정해?" → 문서에 없는 UI 단계를 LLM이 지어냄 → faithfulness 0.286

### 언제 실행하면 유용한가
- knowhow 문서 추가/삭제 후
- 임베딩 모델 변경 후  
- 프롬프트 수정 후
- → `ragas_report.json`에 버전별 점수가 쌓여서 전후 비교 가능

In [ ]:
## ── ragas_eval.py 핵심 구조 ───────────────────────────────────────────────
#
# 실행: python3 ragas_eval.py
# 설치: pip install ragas datasets

import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import os, json
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings as LCEmbeddings
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

load_dotenv()
_lc_llm    = ChatOpenAI(model="gpt-4o-mini",          api_key=os.getenv("OPENAI_API_KEY"))
_lc_emb    = LCEmbeddings(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))
_ragas_llm = LangchainLLMWrapper(_lc_llm)
_ragas_emb = LangchainEmbeddingsWrapper(_lc_emb)

# ragas 인스턴스에 LLM / Embeddings 주입 (ragas 0.4.x 방식)
faithfulness.llm            = _ragas_llm
answer_relevancy.llm        = _ragas_llm
answer_relevancy.embeddings = _ragas_emb
context_precision.llm       = _ragas_llm
context_recall.llm          = _ragas_llm

# ── 평가 데이터 구성 ───────────────────────────────────────────────────────
# ragas 0.4.x 필드 매핑:
#   question     → user_input
#   answer       → response
#   contexts     → retrieved_contexts  (List[str])
#   ground_truth → reference

def build_dataset(records):
    samples = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=r["contexts"],
            reference=r.get("ground_truth"),
        )
        for r in records
    ]
    return EvaluationDataset(samples=samples)

# ── 평가 실행 ──────────────────────────────────────────────────────────────
# result = evaluate(dataset, metrics=[faithfulness, answer_relevancy,
#                                     context_precision, context_recall])
# df = result.to_pandas()   # 질문별 점수 DataFrame
# print(df)

---
## 4. 오늘 발생한 트러블슈팅 기록

실제 실행 중 만난 버그들입니다. 동일한 환경에서 작업할 때 참고하세요.

| # | 에러 | 원인 | 해결 |
|---|---|---|---|
| 1 | `TypeError: Type is not msgpack serializable: Document` | MemorySaver가 LangChain `Document` 객체를 직렬화 못 함 | `retrieve`에서 `Document` → `dict` 변환 후 반환 |
| 2 | `TypeError: Type is not msgpack serializable: numpy.float64` | metadata에 numpy 타입이 섞임 | `float(v) if hasattr(v, "item") else v` 로 변환 |
| 3 | `5건 검색 → 0건 채택` | FAISS L2 distance를 `1-score`로 변환하면 score>1 일 때 0이 됨 | `cosine = 1 - (score²)/2` 공식으로 교체 |
| 4 | `Faithfulness.__init__() missing 1 required positional argument: 'llm'` | ragas 0.4.x에서 메트릭에 LLM 직접 주입 필요 | `LangchainLLMWrapper` 사용 |
| 5 | `TypeError: All metrics must be initialised metric objects` | `ragas.metrics.collections`의 클래스 메트릭이 `evaluate()`와 호환 안 됨 | `ragas.metrics`의 인스턴스에 `.llm` 속성 주입 방식으로 교체 |
| 6 | `AnswerRelevancy.__init__() missing 1 required positional argument: 'embeddings'` | `AnswerRelevancy`는 LLM + embeddings 둘 다 필요 | `LangchainEmbeddingsWrapper`로 embeddings도 주입 |